<a href="https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/04-evals/03-regression-evals.zh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 回归评测

**目标：** 把黄金集和 judge 变成一个 CI 风格的测试骨架：运行、保存、diff、设卡。

本项目是 [AI-EDF-NOTEBOOKS-CN](https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN) 的一部分。


## 环境准备

每个 notebook 都是自包含的，所以接下来两个单元格会从零把它搭起来：

1. **安装依赖。** `aien` 包（就是本仓库）带有共享的环境准备助手，并会拉入 `groq` 客户端；本 notebook 还会用到 `sentence-transformers`、`numpy`。
2. **加载你的 API key。** 到 [console.groq.com](https://console.groq.com/) 领一个免费 key（无需信用卡）。在 Colab 里，通过左侧边栏的 **钥匙图标** → **Add new secret** 添加它，名称必须严格写成 `GROQ_API_KEY`，粘贴值，然后打开 **Notebook access** 开关。在本地运行？把 `GROQ_API_KEY` 设为环境变量。

（完整走查和选模型指引见 [00-setup/00-environment.ipynb](https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/00-setup/00-environment.zh.ipynb)。）

In [ ]:
%pip install -q "git+https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN.git@main" sentence-transformers numpy

In [ ]:
from aien import setup

# 读取 GROQ_API_KEY（Colab Secrets 或本地环境变量），并返回一个就绪的
# Groq 客户端。传 model=... 可覆盖默认值；如果之后某次调用报 404，
# 就列出可用模型 —— 见 00-setup/00-environment.ipynb。
client, MODEL = setup()

> **🔵 撞上速率限制？换模型。** 这是本节里最重的 notebook。它会把整个黄金集跑**两遍**（baseline + candidate），每一遍都是每个 case 一次生成调用*加*一次 judge 调用。Groq 免费档的每分钟 token 上限和每日 token 上限都是**按模型**计算的，所以这里很容易撞上 `429`。两个旋钮，各有各的预算：
> ```python
> client, MODEL = setup(model='openai/gpt-oss-20b')   # 被测系统（生成）
> JUDGE_MODEL = 'qwen/qwen3.6-27b'                     # 在下面的 judge 单元格里设置
> ```
> 让 judge 和被测系统用**不同**的模型。从 [Groq 速率限制页](https://console.groq.com/docs/rate-limits) 挑还有余量的模型；只为验证跑通的话，任何有能力的对话模型都行（避开 agentic 的 `groq/compound*`）。还要更省？在只是检查测试骨架能否跑通时，用 `GOLDEN_SET[:4]` 把集合切小。

## 把评测当作 CI

你改了一个 prompt、一个 chunk size 或一个模型版本。某个东西坏了。唯一的问题是：是评测告诉你的，还是客户告诉你的。

这就是本 notebook 的整个框架：黄金集（notebook 01）加上 judge（notebook 02）变成一套测试套件，每次变更上线前你都要跑它。机制和你早已熟悉的 CI 一模一样：跑套件、存结果、和上一次全绿的运行做 diff、某个门禁失败就挡住合并。唯一新的部分是，单个 case 是有噪声的，所以 diff 和门禁逻辑必须容忍抖动，又不放过真正的回归。


## 被测系统

接下来两个单元格以紧凑形式重建第 03 节的 RAG 系统：下载并清洗十份 IETF RFC，按章节分块，用一个小型本地模型做 embedding，按余弦相似度检索，并带引用作答。

这是一次刻意的复制，而不是 import。每个 notebook 都必须能在全新的 Colab 运行时里从上到下跑通，对教学材料来说，自包含比 DRY 更重要。如果这些细节你不熟，先过一遍 `03-rag/`；这里它们只是我们要评测的对象。


In [ ]:
# 03-rag pipeline 的紧凑版复制。自包含优先于 DRY：本 notebook 必须能
# 在 Colab 里独立跑通。这里每个设计决策的完整走查见 03-rag/。
import os
import re
import urllib.request

RFCS = [791, 793, 1035, 2616, 4271, 5321, 6455, 6749, 7540, 9110]
os.makedirs('data/rfc', exist_ok=True)
for n in RFCS:
    path = f'data/rfc/rfc{n}.txt'
    if not os.path.exists(path):  # 如果前面的 notebook 已经缓存过就跳过
        urllib.request.urlretrieve(f'https://www.rfc-editor.org/rfc/rfc{n}.txt', path)

def clean_rfc(text):
    """去掉换页符和 RFC txt 文件自带的页眉/页脚行。"""
    text = text.replace('\f', '\n')
    lines = [l for l in text.split('\n')
             if not re.search(r'\[Page \d+\]\s*$', l)          # 页脚
             and not re.match(r'^\s*RFC \d+.*\d{4}\s*$', l)]   # 页眉
    return re.sub(r'\n{3,}', '\n\n', '\n'.join(lines))

def chunk_rfc(text, rfc, max_chars=2000):
    """按章节分块：按带编号的标题切分，然后限制块的大小。"""
    parts = re.split(r'\n(?=\d+(?:\.\d+)*\.?\s+[A-Z])', text)
    chunks = []
    for part in parts:
        part = part.strip()
        while len(part) > max_chars:
            cut = part.rfind('\n\n', 0, max_chars)
            cut = cut if cut > 200 else max_chars
            chunks.append({'rfc': rfc, 'text': part[:cut].strip()})
            part = part[cut:].strip()
        if len(part) > 100:
            chunks.append({'rfc': rfc, 'text': part})
    return chunks

chunks = []
for n in RFCS:
    with open(f'data/rfc/rfc{n}.txt') as f:
        chunks += chunk_rfc(clean_rfc(f.read()), n)
print(f'{len(chunks)} chunks from {len(RFCS)} RFCs')

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer('all-MiniLM-L6-v2')
emb = embedder.encode([c['text'] for c in chunks],
                      normalize_embeddings=True, show_progress_bar=True)

def retrieve(query, k=5):
    q = embedder.encode([query], normalize_embeddings=True)[0]
    scores = emb @ q  # 余弦相似度：向量已做单位归一化
    return [chunks[i] for i in np.argsort(-scores)[:k]]

def answer_question(question, k=5):
    """检索 top-k 个块，带引用作答。返回 (answer, hits)。"""
    hits = retrieve(question, k)
    context = '\n\n'.join(f"[RFC {h['rfc']}]\n{h['text']}" for h in hits)
    resp = client.chat.completions.create(
        model=MODEL, max_tokens=1024,
        messages=[
            {'role': 'system',
             'content': ('Answer using ONLY the provided RFC excerpts. Cite the RFC number for '
                         'each claim, e.g. (RFC 9110). If the excerpts do not contain the answer, '
                         'say plainly that the corpus does not cover it. Do not guess.')},
            {'role': 'user', 'content': f'{context}\n\nQuestion: {question}'},
        ],
    )
    return resp.choices[0].message.content, hits

notebook 01 和 02 里黄金集与 judge 的自包含复制（设计理由见那两个 notebook）。


In [ ]:
# 黄金集：针对真实语料写的 ~15 个 case。每个 case 都固定了
# 正确答案必须包含什么（must_mention），以及检索器应当命中
# 哪份 RFC（source_rfc）。无法回答的 case 用 source_rfc=None —— 正确
# 行为是拒答，而幻觉出一个答案是硬失败。
GOLDEN_SET = [
    {'question': 'How does the TCP three-way handshake establish a connection?',
     'reference_answer': 'The initiator sends a SYN carrying its initial sequence number; '
                         'the peer responds with SYN-ACK carrying its own sequence number and '
                         'acknowledging the first; the initiator replies with an ACK. Both '
                         'sides now have synchronized sequence numbers.',
     'source_rfc': 793, 'must_mention': ['SYN', 'ACK', 'sequence'], 'unanswerable': False},
    {'question': 'What does the HTTP 404 status code mean?',
     'reference_answer': '404 Not Found: the origin server did not find a current '
                         'representation for the target resource, or is not willing to '
                         'disclose that one exists.',
     'source_rfc': 9110, 'must_mention': ['404', 'not found'], 'unanswerable': False},
    {'question': 'What are the four roles defined in the OAuth 2.0 framework?',
     'reference_answer': 'Resource owner, resource server, client, and authorization server.',
     'source_rfc': 6749,
     'must_mention': ['resource owner', 'client', 'authorization server', 'resource server'],
     'unanswerable': False},
    {'question': 'What is the difference between an A record and a CNAME record in DNS?',
     'reference_answer': 'An A record maps a name to a 32-bit IPv4 host address; a CNAME '
                         'record maps an alias to the canonical name of its target.',
     'source_rfc': 1035, 'must_mention': ['address', 'canonical'], 'unanswerable': False},
    {'question': 'How does a client upgrade an HTTP connection to a WebSocket connection?',
     'reference_answer': 'The client sends a GET request with Upgrade: websocket and '
                         'Connection: Upgrade headers plus a Sec-WebSocket-Key; the server '
                         'replies 101 Switching Protocols with a Sec-WebSocket-Accept value '
                         'derived from the key.',
     'source_rfc': 6455, 'must_mention': ['Upgrade', '101', 'Sec-WebSocket-Key'],
     'unanswerable': False},
    {'question': 'Which SMTP commands send a mail message, and in what order?',
     'reference_answer': 'MAIL FROM identifies the sender, one or more RCPT TO commands name '
                         'recipients, then DATA introduces the message text, terminated by a '
                         'line containing only a period.',
     'source_rfc': 5321, 'must_mention': ['MAIL', 'RCPT', 'DATA'], 'unanswerable': False},
    {'question': 'What is the purpose of the Time to Live field in the IPv4 header?',
     'reference_answer': 'TTL is an upper bound on datagram lifetime. Every module that '
                         'processes the datagram decrements it, and the datagram is discarded '
                         'when it reaches zero, so packets cannot loop forever.',
     'source_rfc': 791, 'must_mention': ['zero', 'discard'], 'unanswerable': False},
    {'question': 'What is the AS_PATH attribute in BGP and why does it matter?',
     'reference_answer': 'AS_PATH records the sequence of autonomous systems a route '
                         'advertisement has traversed. BGP uses it for loop detection (a '
                         'router rejects routes containing its own AS number) and in route '
                         'selection.',
     'source_rfc': 4271, 'must_mention': ['autonomous system', 'loop'], 'unanswerable': False},
    {'question': 'How does HTTP/2 carry multiple requests over a single connection?',
     'reference_answer': 'HTTP/2 multiplexes exchanges as independent streams over one '
                         'connection. Each frame carries a stream identifier, so concurrent '
                         'requests and responses interleave without blocking each other.',
     'source_rfc': 7540, 'must_mention': ['stream', 'frame'], 'unanswerable': False},
    {'question': 'Which HTTP methods are defined as idempotent?',
     'reference_answer': 'PUT and DELETE, plus all safe methods: GET, HEAD, OPTIONS, TRACE.',
     'source_rfc': 9110, 'must_mention': ['PUT', 'DELETE', 'GET'], 'unanswerable': False},
    {'question': 'Why does a TCP connection enter the TIME-WAIT state, and for how long?',
     'reference_answer': 'The side that closes actively waits in TIME-WAIT for twice the '
                         'maximum segment lifetime (2*MSL), so delayed segments from the old '
                         'incarnation die off before the socket pair can be reused.',
     'source_rfc': 793, 'must_mention': ['TIME-WAIT', 'MSL'], 'unanswerable': False},
    {'question': 'Walk through the OAuth 2.0 authorization code grant flow.',
     'reference_answer': 'The client redirects the resource owner to the authorization '
                         'server; after authentication and consent the server redirects back '
                         'with an authorization code; the client exchanges the code, with its '
                         'own credentials, at the token endpoint for an access token.',
     'source_rfc': 6749, 'must_mention': ['authorization code', 'access token', 'redirect'],
     'unanswerable': False},
    {'question': 'What does HTTP status 503 indicate, and which header can accompany it?',
     'reference_answer': '503 Service Unavailable: the server is currently unable to handle '
                         'the request, due to overload or maintenance. It may send Retry-After '
                         'to indicate how long to wait.',
     'source_rfc': 9110, 'must_mention': ['503', 'Retry-After'], 'unanswerable': False},
    # --- 刻意设为无法回答：正确行为是拒答 ---
    {'question': 'How does QUIC combine the transport and cryptographic handshakes?',
     'reference_answer': 'Declines: QUIC (RFC 9000) is not in the corpus.',
     'source_rfc': None, 'must_mention': [], 'unanswerable': True},
    {'question': 'What cipher suites does TLS 1.3 define?',
     'reference_answer': 'Declines: TLS 1.3 (RFC 8446) is not in the corpus.',
     'source_rfc': None, 'must_mention': [], 'unanswerable': True},
    {'question': 'Which port does IMAP use, and how does a client select a mailbox?',
     'reference_answer': 'Declines: IMAP is not in the corpus (SMTP is the only mail '
                         'protocol here).',
     'source_rfc': None, 'must_mention': [], 'unanswerable': True},
]
print(len(GOLDEN_SET), 'cases,',
      sum(c['unanswerable'] for c in GOLDEN_SET), 'unanswerable')


In [ ]:
def keyword_coverage(case, answer):
    if not case['must_mention']:
        return None
    a = answer.lower()
    return sum(kw.lower() in a for kw in case['must_mention']) / len(case['must_mention'])

DECLINE_MARKERS = ['does not cover', "doesn't cover", 'not covered', 'do not contain',
                   'does not contain', 'not in the corpus', 'cannot answer',
                   'no information', 'not addressed', 'do not include']

def declined(answer):
    a = answer.lower()
    return any(m in a for m in DECLINE_MARKERS)


In [ ]:
import json

# judge 模型：用小档，不用大档。两个原因。(1) 廉价开发、在
# 目标上评测：你在迭代评测骨架本身时会不停地重跑它，所以用
# 便宜模型；在真正的发布决策之前，用更强的模型
# 或人工复核重跑一遍评判。(2) 用与被测系统不同的模型
# 可以避开自我偏好偏差（下面有更多说明）。
JUDGE_MODEL = 'openai/gpt-oss-20b'

JUDGE_TOOL = {
    'type': 'function',
    'function': {
        'name': 'grade_answer',
        'description': 'Record your grade for the candidate answer.',
        'parameters': {
            'type': 'object',
            'properties': {
                'reasoning': {'type': 'string',
                              'description': 'Compare candidate to reference point by point, '
                                             'BEFORE deciding any grade. 2-4 sentences.'},
                'faithful': {'type': 'boolean',
                             'description': 'True if every claim in the candidate is supported '
                                            'by the reference (nothing fabricated).'},
                'complete': {'type': 'boolean',
                             'description': 'True if the candidate covers every key point of '
                                            'the reference.'},
                'score': {'type': 'integer',
                          'description': 'Overall 1-5, anchored to the rubric.'},
            },
            'required': ['reasoning', 'faithful', 'complete', 'score'],
        },
    },
}

JUDGE_SYSTEM = """You grade answers from a retrieval-augmented QA system against a reference answer.

Rubric -- anchor your score to these definitions and use the full range:
1 = wrong or fabricated: contradicts the reference or invents facts
2 = mostly wrong: a relevant fragment, but the main point is missing or incorrect
3 = partially correct: main point present, but a real gap or a minor unsupported claim
4 = correct: matches the reference on every key point; only minor omissions
5 = correct and complete: nothing missing, nothing unsupported

If the reference says the correct behavior is to decline, a candidate that declines
scores 5 and a candidate that invents an answer scores 1.

Write your reasoning FIRST, then decide the score. Judge substance, not style:
a short correct answer outranks a long vague one. Extra length earns nothing."""

from groq import BadRequestError

def judge(question, reference, candidate, tries=3):
    """给一个答案打分，并对 Groq 间歇性的 tool_use_failed 做重试（见 notebook 02）。
    返回 {reasoning, faithful, complete, score}；如果始终没能产出
    一次有效调用，则返回哨兵分数 0，这样一个不稳定的分数会表现为一个失败 case，而不是崩
    溃的运行。
    """
    for _ in range(tries):
        try:
            resp = client.chat.completions.create(
                model=JUDGE_MODEL, max_tokens=700,
                tools=[JUDGE_TOOL],
                tool_choice={'type': 'function', 'function': {'name': 'grade_answer'}},
                messages=[
                    {'role': 'system', 'content': JUDGE_SYSTEM},
                    {'role': 'user', 'content':
                     f'Question: {question}\n\nReference answer:\n{reference}'
                     f'\n\nCandidate answer:\n{candidate}'},
                ],
            )
            return json.loads(resp.choices[0].message.tool_calls[0].function.arguments)
        except BadRequestError as e:
            if 'tool_use_failed' in str(e):
                continue
            raise
    return {'reasoning': '(no valid tool call after retries)',
            'faithful': False, 'complete': False, 'score': 0}

## 测试骨架：`run_eval(config)` → 保存成带时间戳的结果

一个函数接收系统配置，跑完整个集合，返回一个结果 dict：逐 case 的行，加上聚合指标（`hit_rate`、`keyword_coverage`、`judge_score_mean`，以及一个针对无法回答集合的硬性标志）。每次运行都保存成带时间戳的 JSON，放在 `eval_runs/` 里。

`eval_runs/` 应该被 gitignore 掉还是提交进仓库？两边都有道理。提交进来能白得历史和 blame（一个结果文件就放在产生它的那次提交旁边），代价是嘈杂的 diff 和答案进了仓库。gitignore 能让仓库保持干净，但你得另找地方存历史：一个 artifact store，或者最终是一个托管的评测平台。一个合理的折中是：把逐次运行的文件 gitignore 掉，只提交一个小的 `baseline.json` 供门禁对比。


In [ ]:
import json
import time

def run_eval(top_k=5, label='baseline'):
    """用给定配置把黄金集跑过 RAG 系统。
    成本提示：每个 case 一次生成调用（120B）+ 一次 judge 调用（20B）。
    """
    rows = []
    for case in GOLDEN_SET:
        answer, hits = answer_question(case['question'], k=top_k)
        hit = (case['source_rfc'] in [h['rfc'] for h in hits]
               if case['source_rfc'] else None)
        cov = keyword_coverage(case, answer)
        grade = judge(case['question'], case['reference_answer'], answer)
        if case['unanswerable']:
            passed = declined(answer) or grade['score'] >= 4  # judge 能理解拒答
        else:
            passed = bool(hit) and cov >= 0.6 and grade['score'] >= 3
        rows.append({'question': case['question'],
                     'unanswerable': case['unanswerable'],
                     'hit': hit,
                     'coverage': None if cov is None else round(cov, 2),
                     'score': grade['score'],
                     'passed': passed,
                     'answer': answer})
    answerable = [r for r in rows if not r['unanswerable']]
    unanswerable = [r for r in rows if r['unanswerable']]
    results = {
        'label': label,
        'timestamp': time.strftime('%Y%m%dT%H%M%S'),
        'config': {'top_k': top_k, 'model': MODEL, 'judge_model': JUDGE_MODEL},
        'aggregate': {
            'hit_rate': sum(r['hit'] for r in answerable) / len(answerable),
            'keyword_coverage': sum(r['coverage'] for r in answerable) / len(answerable),
            'judge_score_mean': sum(r['score'] for r in answerable) / len(answerable),
            'unanswerable_ok': all(r['passed'] for r in unanswerable),
            'pass_rate': sum(r['passed'] for r in rows) / len(rows),
        },
        'cases': rows,
    }
    os.makedirs('eval_runs', exist_ok=True)
    path = f"eval_runs/{results['timestamp']}_{label}.json"
    with open(path, 'w') as f:
        json.dump(results, f, indent=2)
    print(f'saved {path}')
    for k, v in results['aggregate'].items():
        print(f'  {k:18}: {v if isinstance(v, bool) else round(v, 3)}')
    return results

In [ ]:
# baseline：第 03 节搭出来的系统，top_k=5。
baseline = run_eval(top_k=5, label='baseline_k5')


## 上线一个变更，看它如何坏掉

现在来看那个让整个 notebook 变得具体的演示。我们「上线」一个看起来合理的变更：把 `top_k` 从 5 降到 1，以削减上下文 token 和延迟。这是那种能轻松通过代码评审的变更。代码是对的，demo 查询照样能用，还省钱。评测是它和生产环境之间唯一的屏障。


In [ ]:
# 这个「优化」：只检索最好的那一个块。
# 每 case 再加 2 次调用（120B 生成 + 20B judge）。
candidate = run_eval(top_k=1, label='candidate_k1')

In [ ]:
def compare_runs(a, b):
    """两次评测运行的逐 case diff（同一黄金集、同一顺序）。"""
    print(f"{a['label']}  ->  {b['label']}")
    print(f"{'metric':18} {'before':>8} {'after':>8}")
    for key, va in a['aggregate'].items():
        vb = b['aggregate'][key]
        fa = str(va) if isinstance(va, bool) else f'{va:.3f}'
        fb = str(vb) if isinstance(vb, bool) else f'{vb:.3f}'
        flag = '  <-- ' if fa != fb else ''
        print(f'{key:18} {fa:>8} {fb:>8}{flag}')

    regressions, improvements, unchanged = [], [], []
    for ca, cb in zip(a['cases'], b['cases']):
        if ca['passed'] and not cb['passed']:
            regressions.append(cb)
        elif not ca['passed'] and cb['passed']:
            improvements.append(cb)
        else:
            unchanged.append(cb)

    print(f'\nregressions ({len(regressions)}):  # passed before, fails now')
    for r in regressions:
        print(f"  FAIL hit={r['hit']} cov={r['coverage']} score={r['score']} | "
              f"{r['question'][:55]}")
    print(f'improvements ({len(improvements)}):')
    for r in improvements:
        print(f"  PASS | {r['question'][:60]}")
    print(f'unchanged: {len(unchanged)}')
    return regressions, improvements

regressions, improvements = compare_runs(baseline, candidate)


## 阈值与门禁

diff 需要一个策略才能变成 CI。门禁有两种，严格程度刻意不同，而这种拆分就是整个要点：

| 门禁类型 | 适用于 | 触发逻辑 | 容忍 | 例子 |
|---|---|---|---|---|
| **预算门禁** | 有噪声的聚合指标（平均 judge 分数、命中率） | 只有跌幅超过预算才拦截 | 单个 case 的抖动 | 「平均分不得下跌 > 0.5」 |
| **零容忍门禁** | 绝不能上线的行为 | 任何一次出现都硬失败 | 什么都不容忍 | 「无法回答集合上出现任何幻觉」 |

- **预算门禁**用于有噪声的聚合指标：平均 judge 分数不得下跌超过 0.5，命中率不得下跌超过 10 个百分点。单个 case 会抖（judge 是个模型；生成是非确定性的），所以单个 case 失败不该挡住合并。持续下跌才该。
- **零容忍门禁**用于绝不能上线的失败模式：无法回答集合上出现任何一个幻觉答案都是硬失败，不管平均值看起来多好。一个得 4.8 分、却对自身支撑不了的问题自信编造答案的系统，比一个得 4.2 分、清楚自身边界的系统更糟。

In [ ]:
def check_gates(candidate, baseline, max_score_drop=0.5, max_hit_drop=0.10):
    """返回一个门禁失败列表（为空 = 可以安全上线）。"""
    failures = []
    c, b = candidate['aggregate'], baseline['aggregate']
    if b['judge_score_mean'] - c['judge_score_mean'] > max_score_drop:
        failures.append(f"judge_score_mean dropped "
                        f"{b['judge_score_mean'] - c['judge_score_mean']:.2f} "
                        f"(budget: {max_score_drop})")
    if b['hit_rate'] - c['hit_rate'] > max_hit_drop:
        failures.append(f"hit_rate dropped {b['hit_rate'] - c['hit_rate']:.2f} "
                        f"(budget: {max_hit_drop})")
    if not c['unanswerable_ok']:
        failures.append('hallucinated on the unanswerable set (zero tolerance -- '
                        'hard fail regardless of averages)')
    return failures

failures = check_gates(candidate, baseline)
if failures:
    print('GATE FAILED -- do not ship:')
    for f in failures:
        print('  -', f)
else:
    print('all gates passed')

# 在 CI 里你会用非零退出码，而不是打印：
#   import sys; sys.exit(1 if failures else 0)


## 接入 CI

这个测试骨架已经是 CI 形状了：`run_eval()` 产出 JSON 产物，`check_gates()` 返回通过/失败。把 notebook 的单元格搬进一个脚本（`scripts/run_eval.py`），让它加载一份提交进仓库的 `eval_runs/baseline.json`，跑候选配置，并在门禁失败时非零退出。一个最小的 GitHub Actions 步骤：

```yaml
  eval:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: {python-version: '3.11'}
      - run: pip install groq sentence-transformers numpy
      - run: python scripts/run_eval.py --baseline eval_runs/baseline.json
        env:
          GROQ_API_KEY: ${{ secrets.GROQ_API_KEY }}
      - uses: actions/upload-artifact@v4
        if: always()
        with: {name: eval-results, path: eval_runs/}
```

实操提示：在触及 prompt、检索代码或模型配置的 PR 上跑它（用 `paths:` 过滤器让它避开无关变更）；那 ~32 次 API 调用只花几分钱、几分钟，完全在 PR 反馈预算之内；而当有人正当地改进了系统时，更新提交进仓库的 baseline *就*是评审产物，因为 diff 精确显示了哪些 case 变好了。

## 框架什么时候登场

这个基于文件的测试骨架*就是*概念本身；概念上没有任何缺失。托管评测平台（Braintrust、LangSmith、Langfuse）和开源框架（promptfoo，以及像 ragas 这样的指标库）补的是运维层：用仪表盘代替打印出来的表格，用带趋势的运行历史代替一堆 JSON 文件，跨数周运行追责到单个 case，为人工标注循环提供团队评审队列，以及现成的 CI 集成。当团队壮大到超过你一个人、需要不止一个人看这些运行结果、JSON 目录不再 scalable 时，就该采用一个。你会把这些工具评估得很好，恰恰因为你知道它们在自动化什么。


## 实践与反模式

| ✅ 该做 | ❌ 反模式 |
|---|---|
| 每次改 prompt/chunk/模型前都跑黄金集 | 直接把变更上线，让客户去发现回归 |
| 读**逐 case 的 diff**：哪些问题坏了、为什么 | 只看聚合差值（「分数掉了 0.8」） |
| 对噪声聚合用预算门禁；对绝不能上线的用零容忍 | 一个全局阈值；因单个 case 抖动就挡住合并 |
| 把阈值设在实测噪声底噪之上一点点 | 拍脑袋定阈值；门禁要么乱抖要么形同虚设 |
| 让它足够便宜/快到能在相关 PR 上跑 CI | 一个 500 case 的套件每月跑一次，回归发现得太晚 |
| 也门禁成本/延迟，而不只是质量 | 那种往里塞更多上下文来把分数抬上去的「修复」 |

（贯穿整个技术栈的完整清单见 [docs/best-practices-and-anti-patterns.md](https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/docs/best-practices-and-anti-patterns.md)。）

## 练习

1. **让分块器回归，而不是 top_k。** 给 pipeline 加一个 `chunk_max_chars` 参数，以 400 字符重建块和 embedding（一个确实更差的分块器，会把章节从中间切开），然后对它 `run_eval` 并对比 baseline。它产生的回归指纹和 `top_k=1` 一样吗，还是坏掉的是不同的 case？
2. **度量评测的抖动。** 不做任何改动，把 `run_eval(top_k=5)` 跑三次，两两 diff。在代码零改动的情况下有多少 case 翻面？把你的门禁预算（`max_score_drop`、`max_hit_drop`）设在实测噪声底噪之上一点点。阈值本来就该这么选。
3. **写这个 CI 脚本。** 把测试骨架抽成 `scripts/run_eval.py`，用 argparse（`--top-k`、`--baseline`、`--save-baseline`），由 `check_gates` 决定 0/1 退出。在本地跑两次：一次写 baseline，一次拿 `--top-k 1` 的候选去对门禁。
4. **加一列成本/延迟。** 在 `run_eval` 里记录每个 case 的 `usage.input_tokens`、`usage.output_tokens` 和墙上时钟时间，并加一个门禁：平均输入 token 相对 baseline 不得增长超过 25%。没有成本门禁的质量门禁，会招来那些只会往里塞更多上下文的「修复」。
